# AMR Parsing on Kaggle

## Prerequisites - Upload these as Kaggle Datasets:
1. **amr-code-modules** — `common/` and `model_interface/` folders (use `prepare_kaggle_datasets.py`)
2. **amr-model** — the `mbart-en-id-smaller-concat-finetuned` model files
3. **xlsum-translate-data** — `analysis_data.csv` + `translate/` folder

Enable **GPU accelerator** (T4) in Kaggle notebook settings.

# 1. Create Virtual Environment with Custom Python Version

In [ ]:
%%bash
# ============================================================
# Create a conda virtual environment with a custom Python version
# Change PYTHON_VERSION below to your desired version
# ============================================================
PYTHON_VERSION="3.10"  # <-- Change this to your desired Python version (e.g., 3.9, 3.10, 3.11)
ENV_NAME="amr_env"
ENV_PATH="/kaggle/working/$ENV_NAME"

echo "=== Setting up conda environment with Python $PYTHON_VERSION ==="

# Initialize conda for bash
eval "$(conda shell.bash hook)"

# Create the environment if it doesn't exist
if [ ! -d "$ENV_PATH" ]; then
    conda create -y -p "$ENV_PATH" python=$PYTHON_VERSION
    echo "Environment created at $ENV_PATH"
else
    echo "Environment already exists at $ENV_PATH"
fi

# Activate and install dependencies
conda activate "$ENV_PATH"

echo "Python version: $(python --version)"
echo "Python path: $(which python)"

# Install PyTorch with CUDA support
pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# Install project dependencies
pip install --quiet \
    transformers \
    penman>=1.1.0 \
    sentencepiece \
    sacremoses \
    regex \
    networkx \
    pandas \
    tqdm \
    ipykernel

# Register the environment as a Jupyter kernel
python -m ipykernel install --user --name=$ENV_NAME --display-name "Python ($PYTHON_VERSION) AMR"

echo ""
echo "=== Setup Complete ==="
echo "Python: $(python --version)"
echo "Torch: $(python -c 'import torch; print(torch.__version__)')"
echo "CUDA available: $(python -c 'import torch; print(torch.cuda.is_available())')"
echo ""
echo ">>> IMPORTANT: After this cell finishes, go to Kernel > Change Kernel"
echo ">>>   and select 'Python ($PYTHON_VERSION) AMR' to use the new environment."
echo ">>>   Then continue from Cell 2 below."

## ⚠️ After Cell 1 completes:
1. Go to **Kernel → Change Kernel**
2. Select **"Python (3.10) AMR"** (or whatever version you chose)
3. Then continue running from Cell 2 below

# 2. Setup & Import

In [ ]:
import os, sys, json, torch, penman, numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ============================================================
# KAGGLE DATASET PATHS — adjust the dataset slug names if needed
# ============================================================
CODE_MODULES_PATH = "/kaggle/input/amr-code-modules"
MODEL_PATH        = "/kaggle/input/amr-model"
DATA_PATH          = "/kaggle/input/xlsum-translate-data"
OUTPUT_DIR         = "/kaggle/working/amr_graphs"

# Add code modules to Python path so we can import common/ and model_interface/
if CODE_MODULES_PATH not in sys.path:
    sys.path.insert(0, CODE_MODULES_PATH)

from transformers import AutoConfig
from model_interface.modeling_bart import MBartForConditionalGeneration
from model_interface.tokenization_bart import AMRBartTokenizer
from common.postprocessing import ParsedStatus

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Python Version  : {sys.version}")
print(f"Pytorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

# 3. Load Model & Tokenizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Default Device : {device}")

amr_config = AutoConfig.from_pretrained(MODEL_PATH)
print(f"Model type       : {amr_config.model_type}")
print(f"Architecture     : {amr_config.architectures}")
print(f"Vocab size       : {amr_config.vocab_size}")

amr_tokenizer = AMRBartTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
amr_model = MBartForConditionalGeneration.from_pretrained(MODEL_PATH, config=amr_config)
amr_model.resize_token_embeddings(len(amr_tokenizer))
amr_model = amr_model.to(device)
amr_model.eval()

print(f"Model Loaded on  : {amr_model.device}")
print(f"Model Parameters : {sum(p.numel() for p in amr_model.parameters()):,}")

# 4. Dataset & DataLoader

In [ ]:
class XLSumDataset(Dataset):
    """Custom Dataset For XLSum — Kaggle version.
    
    Only includes rows where:
      1. A translation file exists in the 'translate' folder
      2. An AMR output file does NOT yet exist in the output folder
    """

    def __init__(self, data_path=DATA_PATH, output_dir=OUTPUT_DIR):
        df = pd.read_csv(os.path.join(data_path, "analysis_data.csv"))
        translate_dir = os.path.join(data_path, "translate")

        # Build set of available translation IDs
        available_translations = set()
        if os.path.isdir(translate_dir):
            for fname in os.listdir(translate_dir):
                if fname.endswith(".txt"):
                    available_translations.add(fname[:-4])

        # Build set of already-parsed IDs
        already_parsed = set()
        if os.path.isdir(output_dir):
            for fname in os.listdir(output_dir):
                if fname.endswith(".txt"):
                    already_parsed.add(fname[:-4])

        total_rows = len(df)
        mask_translated = df["id"].isin(available_translations)
        mask_not_parsed = ~df["id"].isin(already_parsed)
        self.df = df[mask_translated & mask_not_parsed].reset_index(drop=True)
        self.translate_dir = translate_dir

        print(f"Total rows in CSV           : {total_rows}")
        print(f"Translations available      : {mask_translated.sum()}")
        print(f"Already parsed (skipped)    : {(mask_translated & ~mask_not_parsed).sum()}")
        print(f"Remaining to parse          : {len(self.df)}")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row["id"] + ".txt"
        with open(os.path.join(self.translate_dir, filename), "r", encoding="utf-8") as f:
            translated = f.read()
        return {
            "id": row["id"],
            "text": row["text"],
            "translated_text": translated
        }

In [ ]:
def add_amr_mask(tokenized_inputs, masks):
    amr_suffix = [
        amr_tokenizer.amr_bos_token_id,
        amr_tokenizer.mask_token_id,
        amr_tokenizer.amr_eos_token_id,
    ]
    amr_mask_suffix = [1, 1, 1]

    updated_inputs, updated_masks = [], []
    for input_ids, mask in zip(tokenized_inputs, masks):
        pad_start = mask.index(0) if 0 in mask else len(mask)
        new_input_ids = input_ids[:pad_start] + amr_suffix + input_ids[pad_start:]
        new_mask = mask[:pad_start] + amr_mask_suffix + mask[pad_start:]
        updated_inputs.append(new_input_ids)
        updated_masks.append(new_mask)
    return updated_inputs, updated_masks


def wrapper_collate_fn(prefix_lang1, prefix_lang2):
    def xlsum_collate_fn(batch):
        all_ids, all_texts, all_translated_texts = [], [], []
        for i in batch:
            all_ids.append(i["id"])
            all_texts.append(i["text"])
            all_translated_texts.append(i["translated_text"])

        all_inputs = []
        for text, translated_text in zip(all_texts, all_translated_texts):
            cur_result = f"{prefix_lang1} {text} {prefix_lang2} {translated_text}"
            all_inputs.append(cur_result)

        tokenized_inputs = amr_tokenizer(
            all_inputs, max_length=None, padding=True, truncation=True
        )
        updated_inputs, updated_masks = add_amr_mask(
            tokenized_inputs["input_ids"],
            tokenized_inputs["attention_mask"]
        )
        return all_ids, updated_inputs, updated_masks
    return xlsum_collate_fn


ds = XLSumDataset()
loader = DataLoader(ds, batch_size=1, collate_fn=wrapper_collate_fn("id_ID", "en_XX"))

# 5. Parsing Functions

In [ ]:
def decode_amr_output(pred_token_ids, tokenizer):
    pred_ids = list(pred_token_ids)
    pred_ids[0] = tokenizer.bos_token_id
    pred_ids = [
        tokenizer.eos_token_id if tok == tokenizer.amr_eos_token_id else tok
        for tok in pred_ids
        if tok != tokenizer.pad_token_id
    ]
    graph, status, (nodes, backreferences) = tokenizer.decode_amr(
        pred_ids, restore_name_ops=False
    )
    amr_string = penman.encode(graph)
    return amr_string, status


def store_graph(ids, amr_strings, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)
    for data_id, amr_str in zip(ids, amr_strings):
        filepath = os.path.join(output_dir, f"{data_id}.txt")
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(amr_str)

# 6. Run AMR Parsing Loop

In [ ]:
status_counts = {"OK": 0, "FIXED": 0, "BACKOFF": 0, "ERROR": 0}
total_parsed = 0

print(f"Starting AMR parsing for {len(ds)} samples...")
print(f"Output directory: {OUTPUT_DIR}")
print(f"{'='*60}\n")

for batch_ids, inputs, masks in tqdm(loader, desc="Parsing AMR"):
    try:
        with torch.no_grad():
            outputs = amr_model.generate(
                input_ids=torch.tensor(inputs, dtype=torch.long).to(device),
                attention_mask=torch.tensor(masks, dtype=torch.long).to(device),
                num_beams=5,
                max_length=1024,
                decoder_start_token_id=amr_tokenizer.amr_bos_token_id
            )

        amr_strings = []
        for i in range(outputs.shape[0]):
            pred_ids = outputs[i].cpu().tolist()
            amr_string, status = decode_amr_output(pred_ids, amr_tokenizer)
            amr_strings.append(amr_string)
            status_name = status.name if hasattr(status, "name") else str(status)
            if status_name in status_counts:
                status_counts[status_name] += 1
            else:
                status_counts["ERROR"] += 1

        store_graph(batch_ids, amr_strings)
        total_parsed += len(batch_ids)

    except Exception as e:
        status_counts["ERROR"] += len(batch_ids)
        continue

    # Periodic GPU memory cleanup
    if total_parsed % 500 == 0:
        torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"PARSING COMPLETE")
print(f"{'='*60}")
print(f"Total parsed : {total_parsed}")
for status_name, count in status_counts.items():
    if count > 0:
        pct = count / max(total_parsed, 1) * 100
        print(f"  {status_name:8s} : {count} ({pct:.1f}%)")
print(f"\nOutput saved to: {OUTPUT_DIR}")

# 7. Zip Output for Download

In [ ]:
import zipfile

zip_path = "/kaggle/working/amr_graphs.zip"
output_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".txt")]

print(f"Zipping {len(output_files)} AMR graph files...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in tqdm(output_files, desc="Zipping"):
        zf.write(os.path.join(OUTPUT_DIR, fname), fname)

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"\nDone! Zip file: {zip_path} ({size_mb:.1f} MB)")
print(f"Total AMR graphs: {len(output_files)}")
print("\nYou can download this from the Output tab on the right.")